# PhaseNet Smoke Test: Ridgecrest Earthquake

This notebook verifies the original PhaseNet v7 weights by running phase picking on SCSN stations near the Ridgecrest earthquake (July 5, 2019, M7.1) and visually inspecting picks against waveforms.

**Goal**: Validate that picks are physically sensible (P arrives before S, timing matches waveforms).

In [1]:
import obspy
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seisbench.models as sbm
import seisbench.util as sbu
from s3fs import S3FileSystem
from tqdm.notebook import tqdm
from datetime import datetime, timedelta

ModuleNotFoundError: No module named 'seisbench'

## Configuration

In [ ]:
# Ridgecrest M7.1 earthquake
EVENT_TIME = obspy.UTCDateTime("2019-07-05T17:33:50Z")  # Approximate
EVENT_LAT = 35.705
EVENT_LON = -117.504

# Time window around event (seconds before/after)
BEFORE_EVENT = 10
AFTER_EVENT = 60

# SCSN stations within 50 km of epicenter
# All from Southern California Seismic Network (CI network)
# Data source: SCEDC S3 bucket (scedc-pds/continuous_waveforms/)
TEST_STATIONS = [
    ("CI", "DAM"),    # Furnace Creek, 6 km NW
    ("CI", "BOR"),    # Boron, 17 km N
    ("CI", "SYC"),    # Sycamore, 34 km NNW
    ("CI", "TNP"),    # Timberlake Peak, 39 km N
    ("CI", "PIG"),    # Pisgah area, 48 km NW
]

# Model settings
P_THRESHOLD = 0.3
S_THRESHOLD = 0.3

## Step 1: Fetch waveforms from NCEDC S3

Download continuous data for the event day.

In [ ]:
fs = S3FileSystem(anon=True)

# Event is on day of year 186 in 2019 (July 5)
year = 2019
doy = 186

# Build S3 paths for HH* channels (broadband high-sample-rate)
# Using SCEDC (Southern California Earthquake Data Center) S3 bucket
waveforms = {}

for net, sta in TEST_STATIONS:
    print(f"\nFetching {net}.{sta}...")
    stream = obspy.Stream()
    
    for channel in "ZNE":
        # SCEDC S3 structure: scedc-pds/continuous_waveforms/{net}/{year}/{year}.{doy:03d}/{sta}.{net}.HH{channel}.00.D.{year}.{doy:03d}
        s3_path = f"scedc-pds/continuous_waveforms/{net}/{year}/{year}.{doy:03d}/{sta}.{net}.HH{channel}.00.D.{year}.{doy:03d}"
        
        try:
            with fs.open(s3_path) as f:
                trace = obspy.read(f)
                stream += trace
                print(f"  ✓ {channel}: {len(trace[0])} samples")
        except FileNotFoundError:
            print(f"  ✗ {channel}: not found")
        except Exception as e:
            print(f"  ✗ {channel}: {type(e).__name__}")
    
    if len(stream) > 0:
        # Trim to 24-hour window starting from first sample
        t0 = min(t.stats.starttime for t in stream)
        stream.trim(starttime=t0, endtime=t0 + 24 * 3600)
        waveforms[f"{net}.{sta}"] = stream
        print(f"  → Stored {len(stream)} traces")
    else:
        print(f"  → Skipped (no data)")

In [2]:
# Quick sanity check: plot one station's raw waveform
if waveforms:
    first_sta = list(waveforms.keys())[0]
    stream = waveforms[first_sta]
    print(f"Sample stream ({first_sta}):")
    print(stream)
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 8))
    for i, channel in enumerate("ZNE"):
        tr = stream.select(channel=f"HH{channel}")
        if tr:
            time_array = np.arange(len(tr[0])) / tr[0].stats.sampling_rate
            axes[i].plot(time_array, tr[0].data, 'k-', linewidth=0.5)
            axes[i].axvline(BEFORE_EVENT, color='r', linestyle='--', alpha=0.7, label="Event time (approx)")
            axes[i].set_ylabel(f"HH{channel}")
            axes[i].legend()
    axes[-1].set_xlabel("Time since 00:00 UTC (s)")
    fig.suptitle(f"{first_sta} - {year}-{doy:03d}")
    plt.tight_layout()
    plt.show()

NameError: name 'waveforms' is not defined

## Step 2: Load PhaseNet original weights (v7)

Load the v7 weights from the lab's phasenet-retrain repo (converted to SeisBench format).

In [ ]:
# Try to load the v7 weights if available
try:
    print("Attempting to load v7 (quakescope2026) weights...")
    model = sbm.PhaseNet.from_pretrained("quakescope2026")
    print(f"✓ Loaded custom v7 weights")
except Exception as e:
    print(f"⚠ Could not load v7 weights: {e}")
    print("  Falling back to SeisBench 'instance' weights for comparison...")
    model = sbm.PhaseNet.from_pretrained("instance")

print(f"\nModel: {model}")
print(f"Model input samples: {model.in_samples}")
print(f"Model output samples: {model.out_samples}")

## Step 3: Run phase picker and collect picks

In [ ]:
all_picks = sbu.PickList()
station_picks = {}  # Store per-station for visualization

for sta_id, stream in tqdm(waveforms.items(), desc="Picking"):
    print(f"\n{sta_id}: {len(stream)} traces, sampling_rate={stream[0].stats.sampling_rate} Hz")
    
    # Run model
    try:
        pred = model.classify(stream, P_threshold=P_THRESHOLD, S_threshold=S_THRESHOLD)
        picks = pred.picks
        print(f"  → {len(picks)} picks: {picks}")
        all_picks += picks
        station_picks[sta_id] = picks
    except Exception as e:
        print(f"  ✗ Error: {e}")

print(f"\n\nTotal picks across all stations: {len(all_picks)}")

## Step 4: Visualize picks on waveforms

Plot a time window around the event with picks overlaid. This is the key validation step—picks should align with waveform features (P and S arrivals).

In [ ]:
def plot_picks_on_waveform(stream, picks, event_time, before=10, after=60, figsize=(14, 8)):
    """
    Plot 3-component waveform with picks overlaid.
    
    Args:
        stream: ObsPy Stream with Z, N, E traces
        picks: SeiBench PickList for this station
        event_time: UTCDateTime of approximate event time
        before, after: seconds to show before/after event time
    """
    fig, axes = plt.subplots(3, 1, figsize=figsize)
    
    channels = ["HHZ", "HHN", "HHE"]
    
    for ax, channel in zip(axes, channels):
        tr = stream.select(channel=channel)
        if not tr:
            ax.text(0.5, 0.5, f"{channel}: no data", ha='center', va='center', transform=ax.transAxes)
            continue
        
        tr = tr[0]
        t_start = tr.stats.starttime
        dt = tr.stats.delta
        
        # Plot full waveform
        time_vec = np.arange(len(tr)) * dt
        ax.plot(time_vec, tr.data, 'k-', linewidth=0.5, alpha=0.7)
        
        # Mark event time
        event_offset = (event_time - t_start)
        ax.axvline(event_offset, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Event time (approx)')
        
        # Overlay picks for this channel
        for pick in picks:
            if channel[2] == pick.phase:  # Match Z/P for P, N/E for S
                pick_offset = (pick.time - t_start)
                pick_prob = pick.peak_value if hasattr(pick, 'peak_value') else 0.5
                
                color = 'blue' if 'P' in str(pick.phase) else 'green'
                ax.axvline(pick_offset, color=color, linewidth=2, alpha=0.8)
                # Add a small marker at the waveform value
                idx = int(pick_offset / dt)
                if 0 <= idx < len(tr):
                    ax.plot(pick_offset, tr.data[idx], marker='o', color=color, markersize=8)
        
        # Highlight event window
        ax.axvspan(event_offset - before, event_offset + after, alpha=0.1, color='gray')
        
        ax.set_ylabel(channel)
        ax.grid(True, alpha=0.3)
        if channel == channels[0]:
            ax.legend(loc='upper right')
    
    axes[-1].set_xlabel("Time since 00:00 UTC (s)")
    fig.suptitle(f"PhaseNet Picks Validation", fontsize=14, fontweight='bold')
    
    return fig

# Plot for each station
for sta_id, picks in station_picks.items():
    stream = waveforms[sta_id]
    print(f"\n{sta_id}: {len(picks)} picks")
    fig = plot_picks_on_waveform(stream, picks, EVENT_TIME, before=BEFORE_EVENT, after=AFTER_EVENT)
    plt.tight_layout()
    plt.show()

## Step 5: Sanity checks

Verify picks make physical sense.

In [ ]:
print("\n=== SANITY CHECKS ===")
print(f"\nTotal picks: {len(all_picks)}")
print(f"Picks per station:")
for sta_id, picks in station_picks.items():
    p_picks = [p for p in picks if 'P' in str(p.phase)]
    s_picks = [p for p in picks if 'S' in str(p.phase)]
    print(f"  {sta_id}: {len(p_picks)} P, {len(s_picks)} S")

print(f"\nPick quality distribution:")
print(all_picks.df.groupby('phase')['peak_value'].describe())

print(f"\n=== P-S Time Differences (Close-field) ===")
print(f"\nExpected P-S delays (Vp≈5.8 km/s, Vs≈3.3 km/s):")
print(f"  DAM (6 km): 0.7-1.5s")
print(f"  BOR (17 km): 2.0-3.2s")
print(f"  SYC (34 km): 4.0-6.0s")
print(f"  TNP (39 km): 4.6-7.0s")
print(f"  PIG (48 km): 5.7-8.6s")

print(f"\nObserved P-S differences:")
for sta_id, picks in station_picks.items():
    df_picks = picks.df
    p_times = df_picks[df_picks['phase'] == 'P']['time'].values
    s_times = df_picks[df_picks['phase'] == 'S']['time'].values
    
    if len(p_times) > 0 and len(s_times) > 0:
        ps_diffs = []
        for p_time in p_times[-5:]:  # Last 5 P picks (likely closest to main event)
            for s_time in s_times[-5:]:
                if s_time > p_time:
                    ps_diffs.append((s_time - p_time))
        if ps_diffs:
            print(f"  {sta_id}: P-S = {min(ps_diffs):.2f} - {max(ps_diffs):.2f} s (mean: {np.mean(ps_diffs):.2f} s)")
        else:
            print(f"  {sta_id}: No valid P-S pairs found")
    else:
        print(f"  {sta_id}: Insufficient picks (P={len(p_times)}, S={len(s_times)})")

## Step 6: Summary and next steps

- **Visual inspection**: Do picks align with waveform onset features?
- **P-S timing**: Does P always precede S with physically reasonable intervals?
- **Magnitude of picks**: Do picks cluster in time near the main event?

If all checks pass, the model is ready for production.

In [ ]:
# Optional: Save picks to file for inspection
# all_picks.to_csv("ridgecrest_picks.csv")
# print("Picks saved to ridgecrest_picks.csv")